# VQA-RAD: Qwen + MedGemma + LLaVA zero-shot comparison

This notebook uses the same VQA-RAD test split for all three models:

- `Qwen/Qwen2.5-VL-3B-Instruct`
- `google/medgemma-4b-it` (loaded from the existing Drive-backed Hugging Face cache)
- `llava-hf/llava-1.5-7b-hf`

Each model produces one greedy answer plus three sampled answers per question. The final cells calculate generation metrics, AFD uncertainty scores, failure AUROC/AUPRC, and selective-prediction metrics.

Run the cells from top to bottom. The original notebook is not modified.


In [ ]:
import subprocess
import sys


def run_pip(*packages, extra_args=()):
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--no-cache-dir",
        *extra_args,
        *packages,
    ]

    print("Running:", " ".join(command))
    subprocess.check_call(command)


# 1. 检查云端预装的 PyTorch 和 GPU
try:
    import torch
    import torchvision

    print("Python:", sys.version)
    print("PyTorch:", torch.__version__)
    print("torchvision:", torchvision.__version__)
    print("PyTorch CUDA build:", torch.version.cuda)
    print("CUDA available:", torch.cuda.is_available())

    if not torch.cuda.is_available():
        raise RuntimeError(
            "PyTorch 已安装，但没有检测到 CUDA。"
            "请先把云端 Runtime 设置为 GPU。"
        )

    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            1,
        ),
        "GB",
    )

except ImportError as error:
    raise RuntimeError(
        "当前云端镜像没有预装 PyTorch/torchvision。"
        "请换成标准 GPU PyTorch 镜像后重新运行。"
    ) from error


# 2. 更新 pip 工具
run_pip(
    "pip",
    "setuptools",
    "wheel",
)


# 3. 安装三个模型和评估所需依赖
# 不再安装或替换 torch/torchvision/torchaudio
run_pip(
    "numpy<2.0",
    "pandas==2.2.2",
    "scipy==1.13.1",
    "scikit-learn==1.5.2",

    "matplotlib",
    "pillow",
    "tqdm",

    "datasets==2.21.0",
    "transformers==4.51.3",
    "tokenizers==0.21.1",
    "huggingface-hub==0.30.2",
    "accelerate==1.6.0",
    "evaluate==0.4.3",

    "qwen-vl-utils",

    "nltk==3.9.1",
    "rouge-score==0.1.2",
    "sacrebleu==2.4.3",

    "sentencepiece",
    "protobuf",
    "einops",
    "safetensors",
    "packaging",
)


# 4. 安装 4-bit 量化工具，但不允许它修改 PyTorch
run_pip(
    "bitsandbytes==0.45.4",
    extra_args=("--no-deps",),
)


print("\n环境安装完成。")
print("现在重启 Runtime / Python Kernel，然后从下一个 cell 继续。")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.11.0+cu128
torchvision: 0.26.0+cu128
PyTorch CUDA build: 12.8
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
GPU memory: 79.3 GB
Running: /usr/bin/python3 -m pip install --upgrade --no-cache-dir pip setuptools wheel
Running: /usr/bin/python3 -m pip install --upgrade --no-cache-dir numpy<2.0 pandas==2.2.2 scipy==1.13.1 scikit-learn==1.5.2 matplotlib pillow tqdm datasets==2.21.0 transformers==4.51.3 tokenizers==0.21.1 huggingface-hub==0.30.2 accelerate==1.6.0 evaluate==0.4.3 qwen-vl-utils nltk==3.9.1 rouge-score==0.1.2 sacrebleu==2.4.3 sentencepiece protobuf einops safetensors packaging
Running: /usr/bin/python3 -m pip install --upgrade --no-cache-dir --no-deps bitsandbytes==0.45.4

环境安装完成。
现在重启 Runtime / Python Kernel，然后从下一个 cell 继续。


In [ ]:
# Run this before loading any Hugging Face model.
import os
from google.colab import drive

DRIVE_MOUNT = "/content/gdrive"
if not os.path.isdir(os.path.join(DRIVE_MOUNT, "MyDrive")):
    drive.mount(DRIVE_MOUNT)

# Reuse the already-downloaded MedGemma and other cached model files.
HF_HOME = "/content/gdrive/MyDrive/hf_cache"
HF_HUB_CACHE = "/content/gdrive/MyDrive/hf_cache/hub"
TRANSFORMERS_CACHE = "/content/gdrive/MyDrive/hf_cache/transformers"
for directory in (HF_HOME, HF_HUB_CACHE, TRANSFORMERS_CACHE):
    os.makedirs(directory, exist_ok=True)

os.environ["HF_HOME"] = HF_HOME
os.environ["HF_HUB_CACHE"] = HF_HUB_CACHE
os.environ["TRANSFORMERS_CACHE"] = TRANSFORMERS_CACHE

print("Drive and Hugging Face cache ready:", HF_HOME)


Mounted at /content/gdrive
Drive and Hugging Face cache ready: /content/gdrive/MyDrive/hf_cache


In [ ]:
# Optional but recommended: verify VQA-RAD before loading any large model.
from datasets import load_dataset

dataset_check = load_dataset("flaviagiammarino/vqa-rad", split="test")
required_columns = {"image", "question", "answer"}
missing_columns = required_columns - set(dataset_check.column_names)
if missing_columns:
    raise RuntimeError(f"VQA-RAD columns missing: {sorted(missing_columns)}")

print("VQA-RAD test size:", len(dataset_check))
print("Columns:", dataset_check.column_names)
print("Example question:", dataset_check[0]["question"])
print("Example answer:", dataset_check[0]["answer"])
del dataset_check


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/1793 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/451 [00:00<?, ? examples/s]

VQA-RAD test size: 451
Columns: ['image', 'question', 'answer']
Example question: is there evidence of an aortic aneurysm?
Example answer: yes


Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

Complete MedGemma saved to:
/content/gdrive/MyDrive/pathvqa_models/google_medgemma_4b_it


In [ ]:
# ============================================================
# VQA-RAD zero-shot inference for Qwen2.5-VL, MedGemma, and LLaVA
# One shared test split; one JSON output per model
# MedGemma uses the existing Drive/Hugging Face cache
# ============================================================

import os
import re
import gc
import json
import math
import random
import warnings

import torch
import numpy as np
import pandas as pd

from PIL import Image
from tqdm.auto import tqdm
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    LlavaForConditionalGeneration,
    Qwen2_5_VLForConditionalGeneration,
    BitsAndBytesConfig,
)
from qwen_vl_utils import process_vision_info

warnings.filterwarnings("ignore")


# ============================================================
# 1. Reproducibility
# ============================================================

SEED = 42
RUN_ID = "vqa_rad_zero_shot_v1"
FORCE_RERUN = False

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


# ============================================================
# 2. Basic configuration
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")


# Use full PathVQA test split
DATASET_NAME = "flaviagiammarino/vqa-rad"
DATASET_TAG = "vqa_rad"
SPLIT_NAME = "test"

# None means use all test samples
MAX_EVAL_SAMPLES = None

# K sampled answers are used for AFD-style uncertainty methods
K_SAMPLED_ANSWERS = 3

# Keep short because PathVQA answers are usually short
MAX_NEW_TOKENS = 32

# Batch size.
# A100: try 4 or 8.
# G4/L4/T4: start with 1 or 2.
DEFAULT_BATCH_SIZE = 4

# Use 4-bit to reduce memory.
# If A100 is stable and you want potentially better speed/quality, you may set this False.
USE_4BIT = True

# Save directory in Google Drive
OUTPUT_DIR = "/content/gdrive/MyDrive/vqa_rad_three_models_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Output directory:", OUTPUT_DIR)


# ============================================================
# 3. System message / prompt
# ============================================================

SYSTEM_MESSAGE = (
    "You are a professional radiologist answering questions about radiology images. "
    "Use only the supplied image and the question. "
    "For yes/no questions, answer only 'yes' or 'no'. "
    "For open-ended questions, return the shortest accurate medical phrase. "
    "Do not repeat the question, explain reasoning, or add unrelated information. "
    "Only output the final answer."
)


def build_user_question(question):
    return (
        f"{SYSTEM_MESSAGE}\n\n"
        f"Question: {question}\n"
        f"Answer:"
    )


def clean_answer(text):
    if text is None:
        return ""

    text = str(text).strip()

    # Remove common assistant prefixes
    text = re.sub(r"^\s*assistant\s*[:：]\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^\s*answer\s*[:：]\s*", "", text, flags=re.IGNORECASE)

    # Keep only first non-empty line
    lines = [x.strip() for x in text.splitlines() if x.strip()]
    text = lines[0] if lines else ""

    # Remove repeated labels
    text = text.replace("Final answer:", "").replace("final answer:", "").strip()

    # Normalize extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ============================================================
# 4. Load PathVQA test split
# ============================================================

vqa_rad_test = load_dataset(
    DATASET_NAME,
    split="test",
)

print("Original VQA-RAD test samples:", len(vqa_rad_test))

if MAX_EVAL_SAMPLES is None:
    eval_hf_dataset = vqa_rad_test
else:
    eval_hf_dataset = vqa_rad_test.select(
        range(min(MAX_EVAL_SAMPLES, len(vqa_rad_test)))
    )

EVAL_SIZE = len(eval_hf_dataset)

print("Using split:", SPLIT_NAME)
print("Evaluation samples:", EVAL_SIZE)


# ============================================================
# 5. Dataset and DataLoader
# ============================================================

class VQARADInferenceDataset(Dataset):
    def __init__(self, hf_dataset):
        self.dataset = hf_dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]

        image = sample["image"].convert("RGB")
        question = str(sample["question"])
        answer = str(sample["answer"])

        return {
            "index": idx,
            "image": image,
            "question": question,
            "answer": answer,
        }


def collate_fn(batch):
    return {
        "index": [x["index"] for x in batch],
        "image": [x["image"] for x in batch],
        "question": [x["question"] for x in batch],
        "answer": [x["answer"] for x in batch],
    }


eval_dataset = VQARADInferenceDataset(eval_hf_dataset)


# ============================================================
# 6. Model list: LLaMA-based + Gemma/MedGemma only
# ============================================================

MODEL_CONFIGS = [
    {
        "model_key": "qwen2_5_vl_3b",
        "model_id": "Qwen/Qwen2.5-VL-3B-Instruct",
        "family": "qwen2_5_vl",
        "batch_size": 8,
        "local_files_only": False,
    },
    {
    "model_key": "medgemma_4b_it",
    "model_id": "google/medgemma-4b-it",
    "family": "medgemma",
    "batch_size": 4,
    "local_path": MEDGEMMA_LOCAL_PATH,
    "local_files_only": True,
    },
    {
        "model_key": "llava_1_5_7b",
        "model_id": "llava-hf/llava-1.5-7b-hf",
        "family": "llava",
        "batch_size": 4,
        "local_files_only": False,
    },
]


# ============================================================
# 7. Helper functions
# ============================================================

def safe_name(model_id):
    return model_id.replace("/", "_").replace(".", "_").replace("-", "_")


def get_records_path(model_id, model_key):
    file_name = (
        f"outputs_{DATASET_TAG}_{model_key}_{safe_name(model_id)}_"
        f"{SPLIT_NAME}_N{EVAL_SIZE}_K{K_SAMPLED_ANSWERS}_{RUN_ID}.json"
    )
    return os.path.join(OUTPUT_DIR, file_name)


def save_records(path, metadata, records):
    payload = {
        "metadata": metadata,
        "records": records,
    }

    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)


def load_existing_records(path, expected_metadata):
    if not os.path.exists(path):
        return []

    try:
        with open(path, "r", encoding="utf-8") as f:
            payload = json.load(f)

        if isinstance(payload, dict) and "records" in payload:
            old_metadata = payload.get("metadata", {})
            records = payload.get("records", [])

            if old_metadata != expected_metadata:
                print("Cache exists but metadata differs. Existing cache will not be used.")
                return []

            print(f"Loaded existing cache: {len(records)} records")
            return records

        if isinstance(payload, list):
            print(f"Loaded old-format cache: {len(payload)} records")
            return payload

    except Exception as e:
        print("Failed to load existing cache:", repr(e))

    return []


def set_padding_side_left(processor):
    if hasattr(processor, "tokenizer") and processor.tokenizer is not None:
        processor.tokenizer.padding_side = "left"

        if processor.tokenizer.pad_token_id is None:
            processor.tokenizer.pad_token = processor.tokenizer.eos_token


def move_inputs_to_device(inputs, device):
    moved = {}

    for k, v in inputs.items():
        if torch.is_tensor(v):
            moved[k] = v.to(device)
        else:
            moved[k] = v

    return moved


def build_prompts_for_family(family, questions):
    prompts = []

    for q in questions:
        user_text = build_user_question(q)

        if family == "llava":
            prompt = f"USER: <image>\n{user_text}\nASSISTANT:"
            prompts.append(prompt)

        elif family == "medgemma":
            messages = [
                {
                    "role": "system",
                    "content": [{"type": "text", "text": SYSTEM_MESSAGE}],
                },
                {
                    "role": "user",
                    "content": [
                        {"type": "image"},
                        {"type": "text", "text": f"Question: {q}\nAnswer:"},
                    ],
                },
            ]
            prompts.append(messages)

        else:
            raise ValueError(f"Unknown family: {family}")

    return prompts


def apply_medgemma_chat_template(processor, messages_list):
    formatted_texts = []

    for messages in messages_list:
        formatted = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        formatted_texts.append(formatted)

    return formatted_texts


def build_qwen_messages(images, questions):
    messages_batch = []
    for image, question in zip(images, questions):
        messages_batch.append(
            [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image},
                        {"type": "text", "text": build_user_question(question)},
                    ],
                }
            ]
        )
    return messages_batch


def generate_batch(
    model,
    processor,
    family,
    images,
    questions,
    do_sample=False,
    max_new_tokens=24,
):
    # ------------------------------------------------------------
    # Case 0: Qwen2.5-VL batched generation
    # ------------------------------------------------------------
    if family == "qwen2_5_vl":
        messages_batch = build_qwen_messages(images, questions)
        formatted_texts = [
            processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
            for messages in messages_batch
        ]
        image_inputs, video_inputs = process_vision_info(messages_batch)
        inputs = processor(
            text=formatted_texts,
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        inputs = move_inputs_to_device(inputs, DEVICE)
        generation_kwargs = {
            "max_new_tokens": max_new_tokens,
            "do_sample": do_sample,
            "pad_token_id": processor.tokenizer.pad_token_id,
        }
        if do_sample:
            generation_kwargs.update({"temperature": 0.7, "top_p": 0.9})
        with torch.no_grad():
            output_ids = model.generate(**inputs, **generation_kwargs)
        generated_ids = [
            output[len(input_ids):]
            for input_ids, output in zip(inputs["input_ids"], output_ids)
        ]
        decoded = processor.batch_decode(
            generated_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )
        return [clean_answer(x) for x in decoded]

    # ------------------------------------------------------------
    # Case 1: MedGemma / Gemma3
    # Use single-sample generation for stability.
    # Gemma3 processor is sensitive to batched image-text formatting.
    # ------------------------------------------------------------
    if family == "medgemma":
        decoded_outputs = []

        for img, question in zip(images, questions):
            prompts = build_prompts_for_family(family, [question])
            texts = apply_medgemma_chat_template(processor, prompts)

            inputs = processor(
                text=texts,
                images=[img],
                padding=True,
                return_tensors="pt",
            )

            inputs = move_inputs_to_device(inputs, DEVICE)

            generation_kwargs = {
                "max_new_tokens": max_new_tokens,
                "do_sample": do_sample,
            }

            if do_sample:
                generation_kwargs.update(
                    {
                        "temperature": 0.7,
                        "top_p": 0.9,
                    }
                )

            if hasattr(processor, "tokenizer") and processor.tokenizer is not None:
                if processor.tokenizer.pad_token_id is not None:
                    generation_kwargs["pad_token_id"] = processor.tokenizer.pad_token_id
                elif processor.tokenizer.eos_token_id is not None:
                    generation_kwargs["pad_token_id"] = processor.tokenizer.eos_token_id

            with torch.no_grad():
                output_ids = model.generate(
                    **inputs,
                    **generation_kwargs,
                )

            input_length = inputs["input_ids"].shape[1]
            generated_ids = output_ids[:, input_length:]

            decoded = processor.batch_decode(
                generated_ids,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True,
            )

            decoded_outputs.append(clean_answer(decoded[0]))

        return decoded_outputs

    # ------------------------------------------------------------
    # Case 2: LLaVA
    # Keep normal batch inference for speed.
    # ------------------------------------------------------------
    prompts = build_prompts_for_family(family, questions)

    inputs = processor(
        text=prompts,
        images=images,
        padding=True,
        return_tensors="pt",
    )

    inputs = move_inputs_to_device(inputs, DEVICE)

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
    }

    if do_sample:
        generation_kwargs.update(
            {
                "temperature": 0.7,
                "top_p": 0.9,
            }
        )

    if hasattr(processor, "tokenizer") and processor.tokenizer is not None:
        if processor.tokenizer.pad_token_id is not None:
            generation_kwargs["pad_token_id"] = processor.tokenizer.pad_token_id
        elif processor.tokenizer.eos_token_id is not None:
            generation_kwargs["pad_token_id"] = processor.tokenizer.eos_token_id

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            **generation_kwargs,
        )

    input_length = inputs["input_ids"].shape[1]
    generated_ids = output_ids[:, input_length:]

    decoded = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )

    decoded = [clean_answer(x) for x in decoded]

    return decoded


def load_model_and_processor(model_config):
    model_id = model_config["model_id"]
    family = model_config["family"]
    configured_local_path = model_config.get("local_path")
    has_local_directory = bool(
        configured_local_path
        and os.path.isfile(os.path.join(configured_local_path, "config.json"))
    )
    model_source = configured_local_path if has_local_directory else model_id
    local_files_only = bool(
        has_local_directory or model_config.get("local_files_only", False)
    )

    print("=" * 80)
    print("Loading model:", model_id)
    print("Family:", family)
    print("4-bit:", USE_4BIT)
    print("Local files only:", local_files_only)
    print("=" * 80)

    processor = AutoProcessor.from_pretrained(
        model_source,
        trust_remote_code=True,
        local_files_only=local_files_only,
    )

    set_padding_side_left(processor)

    quantization_config = None

    if USE_4BIT:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )

    if family == "qwen2_5_vl":
        model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            model_source,
            device_map="auto",
            low_cpu_mem_usage=True,
            quantization_config=quantization_config,
            trust_remote_code=True,
            local_files_only=local_files_only,
        )

    elif family == "llava":
        model = LlavaForConditionalGeneration.from_pretrained(
            model_source,
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            device_map="auto",
            low_cpu_mem_usage=True,
            quantization_config=quantization_config,
            trust_remote_code=True,
            local_files_only=local_files_only,
        )

    elif family == "medgemma":
        model = AutoModelForImageTextToText.from_pretrained(
            model_source,
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            device_map="auto",
            low_cpu_mem_usage=True,
            quantization_config=quantization_config,
            trust_remote_code=True,
            local_files_only=local_files_only,
        )

    else:
        raise ValueError(f"Unknown model family: {family}")

    model.eval()

    print("Model loaded:", model_id)

    return model, processor


# ============================================================
# 8. Main overnight inference loop
# ============================================================

all_model_output_files = []

for model_config in MODEL_CONFIGS:
    model_key = model_config["model_key"]
    model_id = model_config["model_id"]
    family = model_config["family"]
    batch_size = model_config["batch_size"]

    metadata = {
        "dataset": DATASET_NAME,
        "dataset_tag": DATASET_TAG,
        "split": SPLIT_NAME,
        "eval_size": EVAL_SIZE,
        "model_key": model_key,
        "model_id": model_id,
        "family": family,
        "batch_size": batch_size,
        "k_sampled_answers": K_SAMPLED_ANSWERS,
        "max_new_tokens": MAX_NEW_TOKENS,
        "system_message": SYSTEM_MESSAGE,
        "use_4bit": USE_4BIT,
        "seed": SEED,
    }


    records_path = get_records_path(model_id, model_key)

    if FORCE_RERUN:
        print("FORCE_RERUN=True, ignoring old cache and generating new outputs.")
        records = []
    else:
        records = load_existing_records(records_path, metadata)

    done_indices = set(int(r["index"]) for r in records if "index" in r)


    print("=" * 80)
    print("Current model:", model_id)
    print(f"Cached records: {len(done_indices)}/{EVAL_SIZE}")
    print("Save path:", records_path)
    print("=" * 80)

    model, processor = load_model_and_processor(model_config)

    eval_loader = DataLoader(
        eval_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
    )

    total_batches = math.ceil(EVAL_SIZE / batch_size)

    for batch in tqdm(eval_loader, total=total_batches, desc=f"Inference: {model_key}"):
        batch_indices = [int(x) for x in batch["index"]]

        if all(idx in done_indices for idx in batch_indices):
            continue

        active_positions = [
            i for i, idx in enumerate(batch_indices)
            if idx not in done_indices
        ]

        images = [batch["image"][i] for i in active_positions]
        questions = [batch["question"][i] for i in active_positions]
        answers = [batch["answer"][i] for i in active_positions]
        indices = [batch_indices[i] for i in active_positions]

        try:
            greedy_predictions = generate_batch(
                model=model,
                processor=processor,
                family=family,
                images=images,
                questions=questions,
                do_sample=False,
                max_new_tokens=MAX_NEW_TOKENS,
            )

            sampled_predictions_by_k = []

            for _ in range(K_SAMPLED_ANSWERS):
                sampled_predictions = generate_batch(
                    model=model,
                    processor=processor,
                    family=family,
                    images=images,
                    questions=questions,
                    do_sample=True,
                    max_new_tokens=MAX_NEW_TOKENS,
                )
                sampled_predictions_by_k.append(sampled_predictions)

            for i in range(len(indices)):
                sampled_answers = [
                    sampled_predictions_by_k[k][i]
                    for k in range(K_SAMPLED_ANSWERS)
                ]

                record = {
                    "index": int(indices[i]),
                    "question": questions[i],
                    "ground_truth": answers[i],
                    "greedy_prediction": greedy_predictions[i],
                    "sampled_answers": sampled_answers,
                    "model_id": model_id,
                    "model_key": model_key,
                    "family": family,
                }

                records.append(record)
                done_indices.add(int(indices[i]))

            save_records(records_path, metadata, records)

        except RuntimeError as e:
            print("RuntimeError happened:", repr(e))

            if "out of memory" in str(e).lower():
                print("CUDA OOM. Try reducing batch_size to 2 or 1 for this model.")
                torch.cuda.empty_cache()

            save_records(records_path, metadata, records)
            raise e

        except Exception as e:
            print("Error happened:", repr(e))
            print("Saving current progress before continuing/stopping.")
            save_records(records_path, metadata, records)
            raise e

    save_records(records_path, metadata, records)
    all_model_output_files.append(records_path)

    print("=" * 80)
    print("Finished model:", model_id)
    print("Total saved records:", len(records))
    print("Saved to:", records_path)
    print("=" * 80)

    del model
    del processor
    gc.collect()

    if DEVICE == "cuda":
        torch.cuda.empty_cache()


print("All finished.")
print("Saved files:")
for p in all_model_output_files:
    print(p)

Device: cuda
GPU: NVIDIA A100-SXM4-80GB
GPU memory: 79.3 GB
Output directory: /content/gdrive/MyDrive/vqa_rad_three_models_outputs
Original VQA-RAD test samples: 451
Using split: test
Evaluation samples: 451
Loaded existing cache: 451 records
Current model: Qwen/Qwen2.5-VL-3B-Instruct
Cached records: 451/451
Save path: /content/gdrive/MyDrive/vqa_rad_three_models_outputs/outputs_vqa_rad_qwen2_5_vl_3b_Qwen_Qwen2_5_VL_3B_Instruct_test_N451_K3_vqa_rad_zero_shot_v1.json
Loading model: Qwen/Qwen2.5-VL-3B-Instruct
Family: qwen2_5_vl
4-bit: True
Local files only: False


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.53G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen2.5-VL-3B-Instruct


...

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

Model loaded: llava-hf/llava-1.5-7b-hf


Inference: llava_1_5_7b:   0%|          | 0/113 [00:00<?, ?it/s]

Finished model: llava-hf/llava-1.5-7b-hf
Total saved records: 451
Saved to: /content/gdrive/MyDrive/vqa_rad_three_models_outputs/outputs_vqa_rad_llava_1_5_7b_llava_hf_llava_1_5_7b_hf_test_N451_K3_vqa_rad_zero_shot_v1.json
All finished.
Saved files:
/content/gdrive/MyDrive/vqa_rad_three_models_outputs/outputs_vqa_rad_qwen2_5_vl_3b_Qwen_Qwen2_5_VL_3B_Instruct_test_N451_K3_vqa_rad_zero_shot_v1.json
/content/gdrive/MyDrive/vqa_rad_three_models_outputs/outputs_vqa_rad_medgemma_4b_it_google_medgemma_4b_it_test_N451_K3_vqa_rad_zero_shot_v1.json
/content/gdrive/MyDrive/vqa_rad_three_models_outputs/outputs_vqa_rad_llava_1_5_7b_llava_hf_llava_1_5_7b_hf_test_N451_K3_vqa_rad_zero_shot_v1.json


In [ ]:
# ============================================================
# AFD Evaluation for VQA-RAD Qwen + MedGemma + LLaVA outputs
# Read all outputs_*.json and generate comparison tables
# ============================================================

import os
import re
import json
import math
import random
import numpy as np
import pandas as pd
import torch
import evaluate

from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score
from transformers import AutoTokenizer, AutoModel


# ============================================================
# 1. Paths and basic settings
# ============================================================

OUTPUT_DIR = "/content/gdrive/MyDrive/vqa_rad_three_models_outputs"

FAILURE_ROUGE_L_THRESHOLD = 0.2
FAILURE_METEOR_THRESHOLD = 0.1

EMBED_MODEL_ID = "BAAI/bge-small-en-v1.5"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Output directory:", OUTPUT_DIR)
print("Device:", DEVICE)


# ============================================================
# 2. Text normalization
# ============================================================

def normalize_text(text):
    if text is None:
        return ""

    text = str(text).lower().strip()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def normalize_for_frequency(text):
    text = normalize_text(text)

    yes_set = {"yes", "yeah", "yep", "true", "present"}
    no_set = {"no", "nope", "false", "absent"}

    if text in yes_set:
        return "yes"

    if text in no_set:
        return "no"

    return text


# ============================================================
# 3. HuggingFace evaluate metrics
# Same style as your lab notebook
# ============================================================

def get_nlp_mettics(references, hypotheses):
    bleu = evaluate.load("bleu")
    rouge = evaluate.load("rouge")
    meteor = evaluate.load("meteor")

    references_norm = [normalize_text(x) for x in references]
    hypotheses_norm = [normalize_text(x) for x in hypotheses]

    results_bleu = bleu.compute(
        predictions=hypotheses_norm,
        references=references_norm
    )

    results_rouge = rouge.compute(
        predictions=hypotheses_norm,
        references=references_norm
    )

    results_meteor = meteor.compute(
        predictions=hypotheses_norm,
        references=references_norm
    )

    return {
        "BLEU-1": results_bleu["precisions"][0],
        "BLEU-2": results_bleu["precisions"][1],
        "ROUGE-L": results_rouge["rougeL"],
        "METEOR": results_meteor["meteor"],
    }


# ============================================================
# 4. Per-sample metrics
# Use evaluate on one sample at a time to assign failure labels
# ============================================================

rouge_metric = evaluate.load("rouge")
meteor_metric = evaluate.load("meteor")


def compute_single_rougel_meteor(prediction, reference):
    pred = normalize_text(prediction)
    ref = normalize_text(reference)

    if pred == "":
        pred = "empty"

    if ref == "":
        ref = "empty"

    rouge_result = rouge_metric.compute(
        predictions=[pred],
        references=[ref]
    )

    meteor_result = meteor_metric.compute(
        predictions=[pred],
        references=[ref]
    )

    return {
        "rougeL": rouge_result["rougeL"],
        "meteor": meteor_result["meteor"],
    }


# ============================================================
# 5. Safe AUROC / AUPRC
# ============================================================

def safe_auroc(labels, scores):
    labels = np.array(labels)

    if len(np.unique(labels)) < 2:
        return np.nan

    return roc_auc_score(labels, scores)


def safe_auprc(labels, scores):
    labels = np.array(labels)

    if len(np.unique(labels)) < 2:
        return np.nan

    return average_precision_score(labels, scores)


# ============================================================
# 6. Embedding model for semantic similarity
# ============================================================

print("Loading embedding model:", EMBED_MODEL_ID)

embed_tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_ID)
embed_model = AutoModel.from_pretrained(EMBED_MODEL_ID).to(DEVICE)
embed_model.eval()

print("Embedding model loaded.")


def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

    return torch.sum(token_embeddings * input_mask_expanded, dim=1) / torch.clamp(
        input_mask_expanded.sum(dim=1),
        min=1e-9
    )


@torch.no_grad()
def encode_texts(texts, batch_size=64):
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = [normalize_text(x) for x in texts[i:i + batch_size]]

        encoded = embed_tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )

        encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

        output = embed_model(**encoded)
        embeddings = mean_pooling(output, encoded["attention_mask"])
        embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)

        all_embeddings.append(embeddings.cpu())

    return torch.cat(all_embeddings, dim=0).numpy()


def cosine_similarity_matrix(embeddings):
    return np.matmul(embeddings, embeddings.T)


# ============================================================
# 7. AFD scoring methods
# ============================================================

def score_random(records):
    rng = np.random.default_rng(42)
    return rng.random(len(records)).tolist()


def score_afd_frequency(records):
    scores = []

    for r in records:
        sampled = r.get("sampled_answers", [])

        if len(sampled) == 0:
            scores.append(1.0)
            continue

        normalized = [normalize_for_frequency(x) for x in sampled]
        most_common_count = max(normalized.count(x) for x in set(normalized))

        reliability = most_common_count / len(normalized)
        uncertainty = 1.0 - reliability

        scores.append(uncertainty)

    return scores


def score_semantic_afd(records):
    scores = []

    for r in tqdm(records, desc="Semantic AFD"):
        sampled = r.get("sampled_answers", [])

        if len(sampled) <= 1:
            scores.append(0.0)
            continue

        embeddings = encode_texts(sampled, batch_size=len(sampled))
        sim = cosine_similarity_matrix(embeddings)

        upper_values = []

        for i in range(len(sampled)):
            for j in range(i + 1, len(sampled)):
                upper_values.append(sim[i, j])

        consistency = float(np.mean(upper_values)) if upper_values else 1.0
        uncertainty = 1.0 - consistency

        scores.append(float(uncertainty))

    return scores


def score_answer_disagreement(records):
    scores = []

    for r in records:
        sampled = r.get("sampled_answers", [])

        if len(sampled) == 0:
            scores.append(1.0)
            continue

        normalized = [normalize_for_frequency(x) for x in sampled]
        unique_count = len(set(normalized))

        disagreement = (unique_count - 1) / max(len(normalized) - 1, 1)
        scores.append(float(disagreement))

    return scores


def score_question_aligned_entropy(records):
    scores = []

    for r in tqdm(records, desc="Question-aligned entropy"):
        question = r.get("question", "")
        sampled = r.get("sampled_answers", [])

        if len(sampled) == 0:
            scores.append(1.0)
            continue

        texts = [question] + sampled
        embeddings = encode_texts(texts, batch_size=len(texts))

        question_embedding = embeddings[0:1]
        answer_embeddings = embeddings[1:]

        qa_similarities = np.matmul(answer_embeddings, question_embedding.T).reshape(-1)

        # Normalize QA alignment from [-1, 1] to [0, 1]
        qa_alignment = float(np.mean((qa_similarities + 1.0) / 2.0))

        if len(sampled) > 1:
            answer_sim = cosine_similarity_matrix(answer_embeddings)
            pair_values = []

            for i in range(len(sampled)):
                for j in range(i + 1, len(sampled)):
                    pair_values.append(answer_sim[i, j])

            answer_consistency = float(np.mean(pair_values)) if pair_values else 1.0
            answer_consistency = (answer_consistency + 1.0) / 2.0
        else:
            answer_consistency = 1.0

        reliability = qa_alignment * answer_consistency
        uncertainty = 1.0 - reliability

        scores.append(float(uncertainty))

    return scores


# ============================================================
# 8. Selective prediction metrics
# Lower uncertainty means more reliable
# ============================================================

def selective_metrics(records, uncertainty_scores, coverages=(0.5, 0.7, 0.9)):
    df = pd.DataFrame(records).copy()
    df["uncertainty"] = uncertainty_scores

    df = df.sort_values("uncertainty", ascending=True).reset_index(drop=True)

    output = {}

    for coverage in coverages:
        keep_n = int(round(len(df) * coverage))
        keep_n = max(1, keep_n)

        accepted = df.iloc[:keep_n]

        suffix = f"@{int(coverage * 100)}%"

        output[f"Accepted samples {suffix}"] = keep_n
        output[f"Accepted ROUGE-L {suffix}"] = accepted["rougeL"].mean()
        output[f"Accepted METEOR {suffix}"] = accepted["meteor"].mean()
        output[f"Accepted failure rate {suffix}"] = accepted["failure"].mean()

    return output


# ============================================================
# 9. Evaluate one JSON output file
# ============================================================

def evaluate_output_json(json_path):
    print("=" * 80)
    print("Evaluating:", json_path)
    print("=" * 80)

    with open(json_path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    if isinstance(payload, dict) and "records" in payload:
        metadata = payload.get("metadata", {})
        records = payload["records"]
    else:
        metadata = {}
        records = payload

    records = list(records)

    expected_size = metadata.get("eval_size")
    if expected_size is not None and len(records) != int(expected_size):
        raise RuntimeError(
            f"Incomplete inference output: {len(records)}/{expected_size}: {json_path}"
        )

    model_id = metadata.get("model_id", records[0].get("model_id", "unknown") if records else "unknown")
    model_key = metadata.get("model_key", records[0].get("model_key", "unknown") if records else "unknown")
    batch_size = metadata.get("batch_size", "unknown")
    k_sampled = metadata.get("k_sampled_answers", "unknown")

    references = [r.get("ground_truth", "") for r in records]
    hypotheses = [r.get("greedy_prediction", "") for r in records]

    generation_metrics = get_nlp_mettics(
        references=references,
        hypotheses=hypotheses
    )

    print("Generation metrics:", generation_metrics)

    scored_records = []

    for r in tqdm(records, desc="Computing per-sample metrics"):
        item = dict(r)

        m = compute_single_rougel_meteor(
            prediction=item.get("greedy_prediction", ""),
            reference=item.get("ground_truth", ""),
        )

        item["rougeL"] = m["rougeL"]
        item["meteor"] = m["meteor"]

        item["failure"] = int(
            item["rougeL"] < FAILURE_ROUGE_L_THRESHOLD
            and item["meteor"] < FAILURE_METEOR_THRESHOLD
        )

        scored_records.append(item)

    afd_methods = {
        "Random baseline": score_random,
        "AFD frequency": score_afd_frequency,
        "Semantic AFD": score_semantic_afd,
        "Answer disagreement": score_answer_disagreement,
        "Question-aligned entropy": score_question_aligned_entropy,
    }

    summary_rows = []

    y_true = [r["failure"] for r in scored_records]

    for method_name, method_func in afd_methods.items():
        print("Running method:", method_name)

        scores = method_func(scored_records)

        row = {
            "Model": model_key,
            "Model ID": model_id,
            "Method": method_name,
            "Evaluation samples": len(scored_records),
            "Batch size": batch_size,
            "K sampled answers": k_sampled,
            "BLEU-1": generation_metrics["BLEU-1"],
            "BLEU-2": generation_metrics["BLEU-2"],
            "ROUGE-L": generation_metrics["ROUGE-L"],
            "METEOR": generation_metrics["METEOR"],
            "Failure AUROC": safe_auroc(y_true, scores),
            "Failure AUPRC": safe_auprc(y_true, scores),
            "Mean uncertainty": float(np.mean(scores)),
        }

        row.update(
            selective_metrics(
                records=scored_records,
                uncertainty_scores=scores,
                coverages=(0.5, 0.7, 0.9),
            )
        )

        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)

    scored_df = pd.DataFrame(scored_records)

    base_name = os.path.splitext(os.path.basename(json_path))[0]

    summary_path = os.path.join(
        OUTPUT_DIR,
        f"afd_summary_{base_name}.csv"
    )

    scored_path = os.path.join(
        OUTPUT_DIR,
        f"afd_scored_records_{base_name}.csv"
    )

    summary_df.to_csv(summary_path, index=False)
    scored_df.to_csv(scored_path, index=False)

    print("Saved summary:", summary_path)
    print("Saved scored records:", scored_path)

    return summary_df


# ============================================================
# 10. Run all JSON files and merge results
# ============================================================

json_files = [
    os.path.join(OUTPUT_DIR, f)
    for f in os.listdir(OUTPUT_DIR)
    if f.startswith("outputs_") and f.endswith(".json")
]

json_files = sorted(json_files)

print("Found JSON files:")
for p in json_files:
    print(p)

all_summary = []

for json_path in json_files:
    summary_df = evaluate_output_json(json_path)
    all_summary.append(summary_df)

if not all_summary:
    raise RuntimeError(f"No complete outputs_*.json files found in {OUTPUT_DIR}")

final_summary_df = pd.concat(all_summary, ignore_index=True)

# Sort by model and AUPRC
final_summary_df = final_summary_df.sort_values(
    ["Model", "Failure AUPRC"],
    ascending=[True, False]
).reset_index(drop=True)

# Round numeric columns
for col in final_summary_df.columns:
    if final_summary_df[col].dtype.kind in "fc":
        final_summary_df[col] = final_summary_df[col].round(4)

final_summary_path = os.path.join(
    OUTPUT_DIR,
    "final_vqa_rad_three_models_afd_summary.csv"
)

final_summary_df.to_csv(final_summary_path, index=False)

print("=" * 80)
print("Final summary saved to:")
print(final_summary_path)
print("=" * 80)

display(final_summary_df)

Output directory: /content/gdrive/MyDrive/vqa_rad_three_models_outputs
Device: cuda


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


Loading embedding model: BAAI/bge-small-en-v1.5


tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Semantic AFD:   0%|          | 0/451 [00:00<?, ?it/s]

Running method: Answer disagreement
Running method: Question-aligned entropy


Question-aligned entropy:   0%|          | 0/451 [00:00<?, ?it/s]

Saved summary: /content/gdrive/MyDrive/vqa_rad_three_models_outputs/afd_summary_outputs_vqa_rad_qwen2_5_vl_3b_Qwen_Qwen2_5_VL_3B_Instruct_test_N451_K3_vqa_rad_zero_shot_v1.csv
Saved scored records: /content/gdrive/MyDrive/vqa_rad_three_models_outputs/afd_scored_records_outputs_vqa_rad_qwen2_5_vl_3b_Qwen_Qwen2_5_VL_3B_Instruct_test_N451_K3_vqa_rad_zero_shot_v1.csv
Final summary saved to:
/content/gdrive/MyDrive/vqa_rad_three_models_outputs/final_vqa_rad_three_models_afd_summary.csv


,Model,Model ID,Method,Evaluation samples,Batch size,K sampled answers,BLEU-1,BLEU-2,ROUGE-L,METEOR,...,Accepted METEOR @50%,Accepted failure rate @50%,Accepted samples @70%,Accepted ROUGE-L @70%,Accepted METEOR @70%,Accepted failure rate @70%,Accepted samples @90%,Accepted ROUGE-L @90%,Accepted METEOR @90%,Accepted failure rate @90%
0,llava_1_5_7b,llava-hf/llava-1.5-7b-hf,Semantic AFD,451,4,3,0.3841,0.0488,0.4025,0.2042,...,0.2524,0.4912,316,0.5184,0.2602,0.4747,406,0.4384,0.2212,0.5493
1,llava_1_5_7b,llava-hf/llava-1.5-7b-hf,Question-aligned entropy,451,4,3,0.3841,0.0488,0.4025,0.2042,...,0.2248,0.5442,316,0.4920,0.2468,0.5000,406,0.4433,0.2236,0.5443
2,llava_1_5_7b,llava-hf/llava-1.5-7b-hf,AFD frequency,451,4,3,0.3841,0.0488,0.4025,0.2042,...,0.2398,0.5133,316,0.4770,0.2377,0.5158,406,0.4345,0.2203,0.5493
3,llava_1_5_7b,llava-hf/llava-1.5-7b-hf,Answer disagreement,451,4,3,0.3841,0.0488,0.4025,0.2042,...,0.2398,0.5133,316,0.4770,0.2377,0.5158,406,0.4345,0.2203,0.5493
4,llava_1_5_7b,llava-hf/llava-1.5-7b-hf,Random baseline,451,4,3,0.3841,0.0488,0.4025,0.2042,...,0.2139,0.5619,316,0.3930,0.1991,0.5886,406,0.4049,0.2044,0.5788
5,medgemma_4b_it,google/medgemma-4b-it,Question-aligned entropy,451,4,3,0.4018,0.0660,0.5866,0.3105,...,0.3526,0.2876,316,0.6808,0.3642,0.2658,406,0.6349,0.3360,0.3202
6,medgemma_4b_it,google/medgemma-4b-it,Semantic AFD,451,4,3,0.4018,0.0660,0.5866,0.3105,...,0.3735,0.2611,316,0.6653,0.3457,0.3165,406,0.6302,0.3306,0.3350
7,medgemma_4b_it,google/medgemma-4b-it,AFD frequency,451,4,3,0.4018,0.0660,0.5866,0.3105,...,0.3661,0.2788,316,0.6631,0.3453,0.3196,406,0.6298,0.3306,0.3325
8,medgemma_4b_it,google/medgemma-4b-it,Answer disagreement,451,4,3,0.4018,0.0660,0.5866,0.3105,...,0.3661,0.2788,316,0.6631,0.3453,0.3196,406,0.6298,0.3306,0.3325
9,medgemma_4b_it,google/medgemma-4b-it,Random baseline,451,4,3,0.4018,0.0660,0.5866,0.3105,...,0.3198,0.3363,316,0.5819,0.3038,0.3639,406,0.5896,0.3112,0.3645


In [ ]:
# ============================================================
# Clean table for all models × all AFD methods
# Remove Model ID / Evaluation samples / Batch size / K
# Keep Model + Method
# ============================================================

import os
import pandas as pd

OUTPUT_DIR = "/content/gdrive/MyDrive/vqa_rad_three_models_outputs"

if "final_summary_df" not in globals():
    final_summary_path = os.path.join(
        OUTPUT_DIR,
        "final_vqa_rad_three_models_afd_summary.csv"
    )
    final_summary_df = pd.read_csv(final_summary_path)

all_afd_cols = [
    "Model",
    "Method",

    "BLEU-1",
    "BLEU-2",
    "ROUGE-L",
    "METEOR",

    "Failure AUROC",
    "Failure AUPRC",
    "Mean uncertainty",

    "Accepted ROUGE-L @50%",
    "Accepted METEOR @50%",
    "Accepted failure rate @50%",

    "Accepted ROUGE-L @70%",
    "Accepted METEOR @70%",
    "Accepted failure rate @70%",

    "Accepted ROUGE-L @90%",
    "Accepted METEOR @90%",
    "Accepted failure rate @90%",
]

all_afd_clean_df = final_summary_df[all_afd_cols].copy()

# Sort: model first, then strongest AUPRC
all_afd_clean_df = all_afd_clean_df.sort_values(
    ["Model", "Failure AUPRC"],
    ascending=[True, False]
).reset_index(drop=True)

for col in all_afd_clean_df.columns:
    if col not in ["Model", "Method"]:
        all_afd_clean_df[col] = all_afd_clean_df[col].round(4)

display(all_afd_clean_df)

all_afd_clean_path = os.path.join(
    OUTPUT_DIR,
    "clean_vqa_rad_qwen_medgemma_llava_afd_table.csv"
)

all_afd_clean_df.to_csv(all_afd_clean_path, index=False)

print("Saved clean all-AFD table to:")
print(all_afd_clean_path)

,Model,Method,BLEU-1,BLEU-2,ROUGE-L,METEOR,Failure AUROC,Failure AUPRC,Mean uncertainty,Accepted ROUGE-L @50%,Accepted METEOR @50%,Accepted failure rate @50%,Accepted ROUGE-L @70%,Accepted METEOR @70%,Accepted failure rate @70%,Accepted ROUGE-L @90%,Accepted METEOR @90%,Accepted failure rate @90%
0,llava_1_5_7b,Semantic AFD,0.3841,0.0488,0.4025,0.2042,0.6367,0.7353,0.1804,0.5018,0.2524,0.4912,0.5184,0.2602,0.4747,0.4384,0.2212,0.5493
1,llava_1_5_7b,Question-aligned entropy,0.3841,0.0488,0.4025,0.2042,0.6154,0.7284,0.3094,0.4472,0.2248,0.5442,0.4920,0.2468,0.5000,0.4433,0.2236,0.5443
2,llava_1_5_7b,AFD frequency,0.3841,0.0488,0.4025,0.2042,0.5962,0.6569,0.2838,0.4804,0.2398,0.5133,0.4770,0.2377,0.5158,0.4345,0.2203,0.5493
3,llava_1_5_7b,Answer disagreement,0.3841,0.0488,0.4025,0.2042,0.5962,0.6569,0.4257,0.4804,0.2398,0.5133,0.4770,0.2377,0.5158,0.4345,0.2203,0.5493
4,llava_1_5_7b,Random baseline,0.3841,0.0488,0.4025,0.2042,0.5156,0.5965,0.4972,0.4209,0.2139,0.5619,0.3930,0.1991,0.5886,0.4049,0.2044,0.5788
5,medgemma_4b_it,Question-aligned entropy,0.4018,0.0660,0.5866,0.3105,0.6603,0.6147,0.2460,0.6427,0.3526,0.2876,0.6808,0.3642,0.2658,0.6349,0.3360,0.3202
6,medgemma_4b_it,Semantic AFD,0.4018,0.0660,0.5866,0.3105,0.6551,0.5205,0.0347,0.7154,0.3735,0.2611,0.6653,0.3457,0.3165,0.6302,0.3306,0.3350
7,medgemma_4b_it,AFD frequency,0.4018,0.0660,0.5866,0.3105,0.5977,0.4568,0.0717,0.6999,0.3661,0.2788,0.6631,0.3453,0.3196,0.6298,0.3306,0.3325
8,medgemma_4b_it,Answer disagreement,0.4018,0.0660,0.5866,0.3105,0.5977,0.4568,0.1075,0.6999,0.3661,0.2788,0.6631,0.3453,0.3196,0.6298,0.3306,0.3325
9,medgemma_4b_it,Random baseline,0.4018,0.0660,0.5866,0.3105,0.5217,0.3926,0.4972,0.6116,0.3198,0.3363,0.5819,0.3038,0.3639,0.5896,0.3112,0.3645


Saved clean all-AFD table to:
/content/gdrive/MyDrive/vqa_rad_three_models_outputs/clean_vqa_rad_qwen_medgemma_llava_afd_table.csv


In [ ]:
# ============================================================
# Final experiment archive
# 按“模型 + 数据集 + 次数”归档本次所有结果
#
# 放置位置：整个 Notebook 最后一个 Cell
# ============================================================

import os
import json
import shutil
import pandas as pd


# ------------------------------------------------------------
# 1. 输入本次实验次数
# ------------------------------------------------------------

RUN_NUMBER = input(
    "请输入本次实验次数，例如 1、2、3："
).strip()

if not RUN_NUMBER.isdigit() or int(RUN_NUMBER) < 1:
    raise ValueError(
        "实验次数必须是正整数，例如：1"
    )

RUN_NUMBER = str(int(RUN_NUMBER))

DATASET_SAVE_NAME = "vqa_rad"

SAVE_ROOT = os.path.join(
    "/content/gdrive/MyDrive",
    "vqa_rad_three_models_outputs",
)

os.makedirs(
    SAVE_ROOT,
    exist_ok=True,
)

print("Run number:", RUN_NUMBER)
print("Save directory:", SAVE_ROOT)


# ------------------------------------------------------------
# 2. 获得本次三模型原始 JSON 文件
# ------------------------------------------------------------

source_json_files = []

# 多模型推理 Cell 通常会生成这个列表
if (
    "all_model_output_files" in globals()
    and all_model_output_files
):
    source_json_files.extend(
        all_model_output_files
    )

# 如果列表不存在，则从 records_path 获取当前模型结果
if (
    not source_json_files
    and "records_path" in globals()
    and os.path.isfile(records_path)
):
    source_json_files.append(
        records_path
    )

# 最后从旧输出目录中搜索
if not source_json_files:
    candidate_directories = [
        SAVE_ROOT,
        "/content/gdrive/MyDrive/pathvqa_llama_gemma_outputs",
        "/content/gdrive/MyDrive/pathvqa_afd_full_comparison",
    ]

    for directory in candidate_directories:
        if not os.path.isdir(directory):
            continue

        for file_name in os.listdir(directory):
            if (
                file_name.startswith("outputs_")
                and file_name.endswith(".json")
            ):
                source_json_files.append(
                    os.path.join(
                        directory,
                        file_name,
                    )
                )

# 去重并过滤不存在的文件
source_json_files = list(
    dict.fromkeys(
        path
        for path in source_json_files
        if os.path.isfile(path)
    )
)

if not source_json_files:
    raise RuntimeError(
        "没有找到模型推理 JSON。"
        "请先等待三模型推理 Cell 完成。"
    )

print("\nFound inference files:")

for path in source_json_files:
    print("-", path)


# ------------------------------------------------------------
# 3. 读取 metadata，按模型+数据集+次数保存
# ------------------------------------------------------------

archived_json_files = []
archived_model_keys = []

for source_path in source_json_files:

    with open(
        source_path,
        "r",
        encoding="utf-8",
    ) as file:
        payload = json.load(file)

    if isinstance(payload, dict):
        metadata = payload.get(
            "metadata",
            {},
        )

        records_in_file = payload.get(
            "records",
            [],
        )
    else:
        metadata = {}
        records_in_file = payload

    model_key = metadata.get(
        "model_key"
    )

    # 兼容旧 JSON：从第一条记录读取模型名
    if (
        not model_key
        and records_in_file
        and isinstance(records_in_file[0], dict)
    ):
        model_key = records_in_file[0].get(
            "model_key"
        )

    # 再从旧文件名识别模型
    if not model_key:
        lowercase_name = os.path.basename(
            source_path
        ).lower()

        if "qwen" in lowercase_name:
            model_key = "qwen2_5_vl_3b"

        elif "medgemma" in lowercase_name:
            model_key = "medgemma_4b_it"

        elif "llava" in lowercase_name:
            model_key = "llava_1_5_7b"

        else:
            model_key = "unknown_model"

    experiment_name = (
        f"{model_key}_"
        f"{DATASET_SAVE_NAME}_"
        f"{RUN_NUMBER}"
    )

    destination_path = os.path.join(
        SAVE_ROOT,
        experiment_name + ".json",
    )

    # 防止意外覆盖其他实验
    if (
        os.path.isfile(destination_path)
        and os.path.abspath(destination_path)
        != os.path.abspath(source_path)
    ):
        raise FileExistsError(
            f"结果已经存在：{destination_path}\n"
            "如果这是新实验，请输入新的次数。"
        )

    if (
        os.path.abspath(destination_path)
        != os.path.abspath(source_path)
    ):
        shutil.copy2(
            source_path,
            destination_path,
        )

    archived_json_files.append(
        destination_path
    )

    archived_model_keys.append(
        model_key
    )

    print(
        f"Saved {model_key}:",
        destination_path,
    )


# ------------------------------------------------------------
# 4. 保存三模型总汇总表
# ------------------------------------------------------------

if "final_summary_df" in globals():

    final_summary_save_path = os.path.join(
        SAVE_ROOT,
        (
            f"all_models_"
            f"{DATASET_SAVE_NAME}_"
            f"{RUN_NUMBER}_summary.csv"
        ),
    )

    if os.path.exists(final_summary_save_path):
        raise FileExistsError(
            f"汇总结果已经存在："
            f"{final_summary_save_path}"
        )

    final_summary_to_save = (
        final_summary_df.copy()
    )

    final_summary_to_save.insert(
        0,
        "Run number",
        RUN_NUMBER,
    )

    final_summary_to_save.insert(
        1,
        "Dataset",
        DATASET_SAVE_NAME,
    )

    final_summary_to_save.to_csv(
        final_summary_save_path,
        index=False,
    )

    print(
        "Saved complete summary:",
        final_summary_save_path,
    )

else:
    print(
        "final_summary_df 不存在："
        "原始模型 JSON 已保存，"
        "但三模型 AFD 汇总表尚未生成。"
    )


# ------------------------------------------------------------
# 5. 保存精简结果表
# ------------------------------------------------------------

clean_dataframe = None

if "all_afd_clean_df" in globals():
    clean_dataframe = (
        all_afd_clean_df.copy()
    )

elif "clean_summary_df" in globals():
    clean_dataframe = (
        clean_summary_df.copy()
    )

elif "final_summary_df" in globals():
    clean_dataframe = (
        final_summary_df.copy()
    )

if clean_dataframe is not None:

    clean_save_path = os.path.join(
        SAVE_ROOT,
        (
            f"all_models_"
            f"{DATASET_SAVE_NAME}_"
            f"{RUN_NUMBER}_clean.csv"
        ),
    )

    if os.path.exists(clean_save_path):
        raise FileExistsError(
            f"精简结果已经存在："
            f"{clean_save_path}"
        )

    if "Run number" not in clean_dataframe.columns:
        clean_dataframe.insert(
            0,
            "Run number",
            RUN_NUMBER,
        )

    if "Dataset" not in clean_dataframe.columns:
        clean_dataframe.insert(
            1,
            "Dataset",
            DATASET_SAVE_NAME,
        )

    clean_dataframe.to_csv(
        clean_save_path,
        index=False,
    )

    print(
        "Saved clean summary:",
        clean_save_path,
    )


# ------------------------------------------------------------
# 6. 保存本次实验清单
# ------------------------------------------------------------

manifest = {
    "run_number": RUN_NUMBER,
    "dataset": DATASET_SAVE_NAME,
    "models": archived_model_keys,
    "inference_json_files": archived_json_files,
}

manifest_path = os.path.join(
    SAVE_ROOT,
    (
        f"experiment_"
        f"{DATASET_SAVE_NAME}_"
        f"{RUN_NUMBER}_manifest.json"
    ),
)

if os.path.exists(manifest_path):
    raise FileExistsError(
        f"实验清单已经存在：{manifest_path}"
    )

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        manifest,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("\n" + "=" * 80)
print("Experiment archive complete")
print("=" * 80)
print("Dataset:", DATASET_SAVE_NAME)
print("Run number:", RUN_NUMBER)
print("Models:", archived_model_keys)
print("Manifest:", manifest_path)
print("=" * 80)

请输入本次实验次数，例如 1、2、3：1
Run number: 1
Save directory: /content/gdrive/MyDrive/vqa_rad_three_models_outputs

Found inference files:
- /content/gdrive/MyDrive/vqa_rad_three_models_outputs/outputs_vqa_rad_qwen2_5_vl_3b_Qwen_Qwen2_5_VL_3B_Instruct_test_N451_K3_vqa_rad_zero_shot_v1.json
- /content/gdrive/MyDrive/vqa_rad_three_models_outputs/outputs_vqa_rad_medgemma_4b_it_google_medgemma_4b_it_test_N451_K3_vqa_rad_zero_shot_v1.json
- /content/gdrive/MyDrive/vqa_rad_three_models_outputs/outputs_vqa_rad_llava_1_5_7b_llava_hf_llava_1_5_7b_hf_test_N451_K3_vqa_rad_zero_shot_v1.json
Saved qwen2_5_vl_3b: /content/gdrive/MyDrive/vqa_rad_three_models_outputs/qwen2_5_vl_3b_vqa_rad_1.json
Saved medgemma_4b_it: /content/gdrive/MyDrive/vqa_rad_three_models_outputs/medgemma_4b_it_vqa_rad_1.json
Saved llava_1_5_7b: /content/gdrive/MyDrive/vqa_rad_three_models_outputs/llava_1_5_7b_vqa_rad_1.json
Saved complete summary: /content/gdrive/MyDrive/vqa_rad_three_models_outputs/all_models_vqa_rad_1_summary.csv
Saved 